# ScriptConsist: model runs on Kaggle

**Before running:**
1. *Settings → Accelerator → GPU T4 x2*, and *Settings → Internet → On*.
2. *Add data*: your uploaded dataset **scriptconsist-code** (the zip from the repo).
3. *Add-ons → Secrets*: add `HF_TOKEN` (a Hugging Face read token), and attach it to this notebook.
4. On huggingface.co, accept the licence on the model pages for **Llama 3.1 8B Instruct** and **Aya Expanse 8B** (one click each; approval is usually quick).

Run cells top to bottom. Each model run is **resumable**: if the session stops, run the same cell again and it continues where it left off.
When everything is done, the last cell makes `results.zip` in the Output panel. Download it and send it back.

**Full run:** after the quick check looks right, click *Save Version → Save & Run All (Commit)*. It runs every cell in the background (about 5–6 hours) and puts `results.zip` in that version's Output tab.

In [ ]:
import os, shutil, glob
WORK = '/kaggle/working/scriptconsist'
hits = glob.glob('/kaggle/input/**/code/run_generation.py', recursive=True)
assert hits, 'Dataset not attached'
SRC = os.path.dirname(os.path.dirname(hits[0]))
os.makedirs(WORK, exist_ok=True)
for sub in ('code', 'data'):   # copy code and data only; never touch results/
    shutil.copytree(os.path.join(SRC, sub), os.path.join(WORK, sub), dirs_exist_ok=True)
os.chdir(WORK)
assert '--prompt-format' in open('code/run_generation.py').read(), 'Old code version attached - update the dataset to the new zip'
print('results so far:', os.listdir('results/generations') if os.path.exists('results/generations') else 'none')

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes rapidfuzz aksharamukha
import torch; print(torch.__version__, torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret('HF_TOKEN'))

## 1. Quick check (about 5 minutes)
Ten questions on Qwen. Read the printed answers: they should be short and in a sensible script. If they look broken, stop and send me the output.

In [ ]:
!python code/run_generation.py --model Qwen/Qwen2.5-7B-Instruct --precision fp16 --limit 10
!python code/score.py results/generations/Qwen__Qwen2.5-7B-Instruct__fp16__limit10.jsonl
import json
try:
    rows=[json.loads(l) for l in open('results/scored/Qwen__Qwen2.5-7B-Instruct__fp16__limit10.jsonl')]
    for r in rows[:25]: print(r['form'], r['prompt_id'], r['correct'], '|', r['response'][:100])
except FileNotFoundError:
    print('Quick check produced no output - send me the messages above')

## 2. Full runs
About 25–40 minutes per model at fp16 and longer at nf4. Run one cell at a time; each is resumable.
The fp16 runs are the main results. The two nf4 runs are the quantization comparison.

In [ ]:
!python code/run_generation.py --model Qwen/Qwen2.5-7B-Instruct --precision fp16
!rm -rf ~/.cache/huggingface/hub   # free disk before the next model

In [ ]:
!python code/run_generation.py --model meta-llama/Llama-3.1-8B-Instruct --precision fp16
!rm -rf ~/.cache/huggingface/hub   # free disk before the next model

In [ ]:
!python code/run_generation.py --model CohereLabs/aya-expanse-8b --precision fp16
!rm -rf ~/.cache/huggingface/hub   # free disk before the next model

In [ ]:
!python code/run_generation.py --model nvidia/Nemotron-4-Mini-Hindi-4B-Instruct --precision fp16
!rm -rf ~/.cache/huggingface/hub   # free disk before the next model

### Hindi/Hinglish-specialised models built on Llama
Both are fine-tuned from Sarvam's **OpenHathi** (Llama 2 7B extended with Hindi and Hinglish).
- **Gajendra v0.1** (BhabhaAI): instruction-tuned on Hindi, Hinglish and English.
- **Airavata** (AI4Bharat): instruction-tuned on Hindi.

Older models sometimes ship without a chat format, so run the check cell first. If it says `chat template present: False`, use the `--prompt-format` shown in the run cell. If the output looks wrong, send it to me before the full run.

In [ ]:
!python code/run_generation.py --model BhabhaAI/Gajendra-v0.1 --show-template

In [ ]:
# if the check above printed 'chat template present: True', keep --prompt-format chat; otherwise try tulu
!python code/run_generation.py --model BhabhaAI/Gajendra-v0.1 --precision fp16 --prompt-format chat --limit 10
import json
try:
    for l in list(open('results/generations/BhabhaAI__Gajendra-v0.1__fp16__limit10.jsonl'))[:12]:
        r = json.loads(l); print(r['form'], '|', r['response'][:100])
except FileNotFoundError:
    print('Gajendra check produced no output - send me the messages above')

In [ ]:
!python code/run_generation.py --model BhabhaAI/Gajendra-v0.1 --precision fp16 --prompt-format chat
!rm -rf ~/.cache/huggingface/hub   # free disk before the next model

In [ ]:
!python code/run_generation.py --model ai4bharat/Airavata --show-template --prompt-format tulu

In [ ]:
!python code/run_generation.py --model ai4bharat/Airavata --precision fp16 --prompt-format tulu
!rm -rf ~/.cache/huggingface/hub   # free disk before the next model

### Quantization comparison (same questions, 4-bit)

In [ ]:
!python code/run_generation.py --model Qwen/Qwen2.5-7B-Instruct --precision nf4
!rm -rf ~/.cache/huggingface/hub   # free disk before the next model

In [ ]:
!python code/run_generation.py --model meta-llama/Llama-3.1-8B-Instruct --precision nf4
!rm -rf ~/.cache/huggingface/hub   # free disk before the next model

### Optional: Gemma 2 9B
Gemma 2 is unstable in fp16, so it runs in 4-bit with full-precision compute. Report it separately from the fp16 models.

In [ ]:
!python code/run_generation.py --model google/gemma-2-9b-it --precision nf4 --batch-size 8
!rm -rf ~/.cache/huggingface/hub   # free disk before the next model

## 3. Score and package

In [ ]:
full=[p for p in glob.glob('results/generations/*.jsonl') if 'limit' not in p]
!python code/score.py {' '.join(full)}
shutil.make_archive('/kaggle/working/results', 'zip', 'results')
print('Download results.zip from the Output panel')